In [ ]:
with tenure as (
    select d.department_name,
    e.name as employee_name,
    e.hire_date,
    max(e.hire_date) over(partition by e.department_id) as most_recent_hire_date,
    rank() over (partition by e.department_id order by e.hire_date asc) as longest_serving_rank
    from employee e
    join department d on e.department_id = d.department_id
),
gaps as (
    select department_name,
    employee_name,
    hire_date,
    longest_serving_rank,
    ((strftime('%Y', most_recent_hire_date) - strftime('%Y', hire_date)) * 12) +
    (strftime('%m', most_recent_hire_date) - strftime('%m', hire_date))
    - (case when strftime('%d', most_recent_hire_date) < strftime('%d', hire_date) then 1 else 0 END) as months_to_most_recent_hire
    from tenure
)
select department_name,
employee_name,
hire_date,
months_to_most_recent_hire
from gaps
where longest_serving_rank = 1;